# Notebook 15 — Mini-Projects and Module Assessment

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 15 of 15  
**Time estimate:** 3–4 hours (implement before reading reference implementations)

> This is the assessment. Every function below is a `NotImplementedError` stub.
> Implement each function, run the evaluation cells, and only then check the
> reference implementations in the HTML comment block at the end.

---
## Mini-Project 1 — Cancer Subtype Classification

**Task:** Build a complete ML pipeline for cancer subtype classification from
simulated RNA-seq data (5 subtypes, 500 samples, 2000 genes). Your pipeline must:
1. Perform PCA-based dimensionality reduction
2. Select the best classifier via nested cross-validation (outer 5-fold, inner 3-fold)
3. Evaluate using accuracy, per-class F1, and confusion matrix
4. Report which genes (by loading magnitude on top PCs) are most discriminating

**Deliverable:** The 4-panel figure (PCA / confusion matrix / CV scores / feature importance).
The confusion matrix must be generated from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# ---- Data generation (do not modify) ----
rng = np.random.default_rng(0)
n_samples, n_genes, n_subtypes = 500, 2000, 5
subtype_means = np.zeros((n_subtypes, n_genes))
for i in range(n_subtypes):
    subtype_means[i, i*80:(i+1)*80] = 3.0
    subtype_means[i, 400+i*30:400+i*30+15] = -2.0

X_rna = np.vstack([
    rng.normal(subtype_means[i], 0.9, (n_samples//n_subtypes, n_genes))
    for i in range(n_subtypes)
])
y_cancer = np.repeat(np.arange(n_subtypes), n_samples//n_subtypes)
print(f'Data: {X_rna.shape}, {n_subtypes} subtypes')

In [ ]:
# ---- MP1: Implement these functions ----

def confusion_matrix_multiclass(y_true, y_pred, n_classes):
    """
    Returns (n_classes x n_classes) confusion matrix.
    Entry [i, j] = number of samples with true label i predicted as j.
    """
    raise NotImplementedError

def per_class_f1(cm):
    """
    Given confusion matrix cm, return array of per-class F1 scores.
    F1_c = 2*TP_c / (2*TP_c + FP_c + FN_c)
    """
    raise NotImplementedError

def nested_cv_classify(X, y, param_grid, outer_k=5, inner_k=3):
    """
    Nested CV: outer StratifiedKFold(outer_k) + inner GridSearchCV(inner_k).
    Pipeline: StandardScaler -> PCA(50) -> LogisticRegression.
    param_grid: dict of LR hyperparams, e.g. {'logisticregression__C': [0.01, 0.1, 1, 10]}
    Returns: array of outer-fold accuracy scores
    """
    raise NotImplementedError

def top_gene_loadings(pca, n_top=20):
    """
    For PC1 and PC2, return the indices of the n_top genes with largest |loading|.
    pca: fitted sklearn PCA object.
    Returns: (top_pc1_indices, top_pc2_indices)
    """
    raise NotImplementedError

In [ ]:
# ---- MP1 Evaluation (run after implementing above) ----
try:
    # Nested CV
    param_grid = {'logisticregression__C': [0.01, 0.1, 1, 10]}
    outer_scores = nested_cv_classify(X_rna, y_cancer, param_grid)
    print(f'Nested CV accuracy: {outer_scores.mean():.3f} ± {outer_scores.std():.3f}')
    
    # Final model for confusion matrix
    pipe = Pipeline([('scaler', StandardScaler()), ('pca', PCA(50)),
                       ('logisticregression', LogisticRegression(C=1, max_iter=2000, multi_class='multinomial'))])
    # Use last outer fold predictions
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    y_pred_all = np.full_like(y_cancer, -1)
    for tr, te in skf.split(X_rna, y_cancer):
        pipe.fit(X_rna[tr], y_cancer[tr])
        y_pred_all[te] = pipe.predict(X_rna[te])
    
    cm = confusion_matrix_multiclass(y_cancer, y_pred_all, n_subtypes)
    f1s = per_class_f1(cm)
    print(f'Per-class F1: {f1s.round(3)}')
    
    pca_fit = PCA(50).fit(StandardScaler().fit_transform(X_rna))
    top_pc1, top_pc2 = top_gene_loadings(pca_fit)
    
    # Portfolio figure
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    cmap = plt.cm.get_cmap('tab10', n_subtypes)
    
    # PCA scatter
    ax = axes[0]
    X_2d = pca_fit.transform(StandardScaler().fit_transform(X_rna))
    for c in range(n_subtypes):
        mask = y_cancer == c
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[cmap(c)], s=15, alpha=0.7, label=f'S{c}')
    ax.set_title('A. PCA')
    ax.legend(fontsize=8)
    
    # Confusion matrix
    ax = axes[1]
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('B. Confusion matrix')
    
    # Nested CV scores per fold
    ax = axes[2]
    ax.bar(range(len(outer_scores)), outer_scores, color='steelblue', edgecolor='black')
    ax.axhline(outer_scores.mean(), color='tomato', ls='--', lw=1.5, label=f'Mean={outer_scores.mean():.3f}')
    ax.set_xlabel('Outer fold'); ax.set_ylabel('Accuracy')
    ax.set_title('C. Nested CV')
    ax.legend(fontsize=8)
    
    # Per-class F1
    ax = axes[3]
    ax.bar(range(n_subtypes), f1s, color=[cmap(c) for c in range(n_subtypes)], edgecolor='black')
    ax.set_xlabel('Subtype'); ax.set_ylabel('F1')
    ax.set_title('D. Per-class F1')
    ax.set_xticks(range(n_subtypes))
    ax.set_xticklabels([f'S{c}' for c in range(n_subtypes)])
    
    plt.suptitle('MP1: Cancer Subtype Classification', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('mp1_cancer_classification.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('MP1 PASSED')
except NotImplementedError:
    print('MP1: Implement the functions above first')

---
## Mini-Project 2 — Drug-Target Binding Prediction

**Task:** Binary classification: does a drug-target pair bind?
Features: 5 physicochemical drug features + 10 protein pocket features.
Requirements:
1. Handle class imbalance (use `class_weight='balanced'` or oversampling)
2. Nested CV for model selection
3. Report AUROC and AUPRC (from scratch)
4. Plot the PR curve and the ROC curve
5. Identify the top 3 features by SHAP or RF importance

In [ ]:
# ---- Data generation (do not modify) ----
rng = np.random.default_rng(1)
n_pairs = 1000
# Features: MW, logP, HBA, HBD, RotBonds | PocketVol, Hydrophobicity, Polarity, FlexScore, BindingPatch x2, Depth, Width, Charge, Conservation
X_drug = np.column_stack([
    rng.normal(400, 80, n_pairs),   # MW
    rng.normal(2, 1.5, n_pairs),    # logP
    rng.integers(0, 8, n_pairs).astype(float),  # HBA
    rng.integers(0, 5, n_pairs).astype(float),  # HBD
    rng.integers(0, 10, n_pairs).astype(float), # RotBonds
])
X_prot = np.column_stack([
    rng.normal(500, 100, n_pairs),  # PocketVol
    rng.normal(0.5, 0.2, n_pairs),  # Hydrophobicity
    rng.normal(0.4, 0.15, n_pairs), # Polarity
    rng.normal(0.6, 0.2, n_pairs),  # FlexScore
    rng.normal(0.7, 0.15, n_pairs), # BindingPatch1
    rng.normal(0.3, 0.2, n_pairs),  # BindingPatch2
    rng.normal(15, 3, n_pairs),     # Depth
    rng.normal(10, 2, n_pairs),     # Width
    rng.normal(0.0, 0.5, n_pairs),  # Charge
    rng.normal(0.5, 0.2, n_pairs),  # Conservation
])
X_binding = np.hstack([X_drug, X_prot])

# True rule (hidden): binding if logP in [1,4] AND PocketVol > 450 AND Hydrophobicity > 0.5
bind_signal = (X_drug[:,1] > 1) & (X_drug[:,1] < 4) & (X_prot[:,0] > 450) & (X_prot[:,1] > 0.5)
y_binding = (bind_signal | rng.random(n_pairs) < 0.05).astype(int)  # 5% noise positives
print(f'Binding dataset: {n_pairs} pairs, {y_binding.mean():.1%} binding')
feat_names = ['MW','logP','HBA','HBD','RotBonds','PocketVol','Hydrophobicity',
                'Polarity','FlexScore','BindPatch1','BindPatch2','Depth','Width','Charge','Conservation']

In [ ]:
# ---- MP2: Implement these functions ----

def roc_curve_scratch(y_true, y_scores):
    """Return (fpr, tpr, auroc)."""
    raise NotImplementedError

def pr_curve_scratch(y_true, y_scores):
    """Return (recall, precision, auprc)."""
    raise NotImplementedError

def nested_cv_binary(X, y, outer_k=5, inner_k=3):
    """
    Nested CV for binary RF classification.
    Inner: GridSearchCV over {'n_estimators': [50, 100], 'max_depth': [3, 5, None]}
    Returns: (outer_auroc_scores, outer_auprc_scores)
    """
    raise NotImplementedError

def rf_feature_importance(X, y, feat_names):
    """
    Fit RF with class_weight='balanced' on full data.
    Return top-3 (feature_name, importance) tuples sorted by descending importance.
    """
    raise NotImplementedError

In [ ]:
# ---- MP2 Evaluation ----
try:
    auroc_scores, auprc_scores = nested_cv_binary(X_binding, y_binding)
    print(f'Nested CV AUROC: {auroc_scores.mean():.3f} ± {auroc_scores.std():.3f}')
    print(f'Nested CV AUPRC: {auprc_scores.mean():.3f} ± {auprc_scores.std():.3f}')
    print(f'Random AUPRC baseline: {y_binding.mean():.3f}')
    
    top3 = rf_feature_importance(StandardScaler().fit_transform(X_binding), y_binding, feat_names)
    print(f'\nTop 3 features: {[(n, f"{i:.4f}") for n, i in top3]}')
    
    # Final model curves
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_binding, y_binding, test_size=0.2,
                                                  random_state=42, stratify=y_binding)
    rf_final = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf_final.fit(StandardScaler().fit_transform(X_tr), y_tr)
    y_prob = rf_final.predict_proba(StandardScaler().fit(X_tr).transform(X_te))[:, 1]
    
    fpr, tpr, auroc = roc_curve_scratch(y_te, y_prob)
    rec, prec, auprc = pr_curve_scratch(y_te, y_prob)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    ax = axes[0]
    ax.plot(fpr, tpr, 'tomato', lw=2, label=f'AUROC={auroc:.3f}')
    ax.plot([0,1],[0,1], 'k--', lw=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('A. ROC curve')
    ax.legend(fontsize=9)
    
    ax = axes[1]
    ax.plot(rec, prec, 'steelblue', lw=2, label=f'AUPRC={auprc:.3f}')
    ax.axhline(y_te.mean(), color='k', ls='--', lw=0.8, label=f'Random={y_te.mean():.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('B. PR curve')
    ax.legend(fontsize=9)
    
    ax = axes[2]
    fi = rf_final.feature_importances_
    order = np.argsort(fi)[-10:]
    ax.barh(range(10), fi[order], color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_yticks(range(10))
    ax.set_yticklabels([feat_names[i] for i in order], fontsize=8)
    ax.set_xlabel('Feature importance')
    ax.set_title('C. RF feature importance\n(top 10)')
    
    plt.suptitle('MP2: Drug-Target Binding Prediction', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('mp2_drug_target.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('MP2 PASSED')
except NotImplementedError:
    print('MP2: Implement the functions above first')

---
## Module Assessment — Part B: Conceptual Questions (60 points)

Answer each question in 2–5 sentences. For derivations, show the algebra.

---
### B1 — Core ML Concepts (20 points)

1. **(4 pts)** Decompose the expected prediction error for a regression model
   into bias, variance, and irreducible noise. For each term, describe one
   hyperparameter that primarily controls it.

2. **(4 pts)** You have a logistic regression model trained with L2 regularization.
   Write the loss function (including the regularization term). Explain why L2
   does not produce sparse solutions whereas L1 does. Use geometry to argue.

3. **(4 pts)** Explain the kernel trick for SVMs. Why can a linear decision boundary
   in a transformed feature space $\phi(x)$ be equivalent to a non-linear boundary
   in the original space $x$? Give the mathematical form of the RBF kernel.

4. **(4 pts)** In gradient boosting, what are pseudo-residuals and why are they used?
   Show that for squared error loss, the pseudo-residuals are exactly the ordinary
   residuals $y_i - F_{m-1}(x_i)$.

5. **(4 pts)** What is nested cross-validation? Draw a diagram of the data splits
   for 5-outer / 3-inner nested CV. Explain why unnested CV with hyperparameter
   tuning gives an optimistically biased performance estimate.

---
### B2 — Unsupervised Learning (15 points)

1. **(5 pts)** Derive the k-means update rule. Show that the centroid update
   $\mu_j = \text{mean}(\text{points in cluster } j)$ is the exact minimizer
   of the within-cluster sum of squares given fixed assignments.

2. **(5 pts)** Describe PCA via SVD in 4 steps. What do the columns of $V$
   represent biologically in an RNA-seq context? What does the scree plot tell you?

3. **(5 pts)** Compare t-SNE and UMAP. What does each method preserve?
   Why should you not interpret distances between clusters in a UMAP plot?

---
### B3 — Evaluation and Clinical Context (15 points)

1. **(5 pts)** A pathogenic variant classifier achieves 99.8% accuracy on a dataset
   where 0.2% of variants are pathogenic. Compute the classifier's precision and
   recall if it predicts 'benign' for everything. Explain why AUPRC is more
   informative than AUROC for this task.

2. **(5 pts)** Write the Cox PH hazard function. What does the C-index measure?
   If a model has C = 0.72, interpret this value in plain language.

3. **(5 pts)** Describe three specific ways that clinical ML problems differ
   from standard benchmark ML problems. For each, describe a concrete mitigation.

---
### B4 — Sequence ML (10 points)

1. **(5 pts)** Explain the difference between one-hot encoding and k-mer features
   for DNA sequences. What information does each representation capture and lose?
   For a 30-mer sequence with $k = 4$, give the dimensions of each feature vector.

2. **(5 pts)** What is a Position Weight Matrix? Write the scoring formula.
   What is information content, and what does IC = 1.8 bits at a position mean
   biologically?

---
## Reference Implementations

<!--
REFERENCE IMPLEMENTATIONS — Read only after attempting

# MP1 References

def confusion_matrix_multiclass(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def per_class_f1(cm):
    n_classes = cm.shape[0]
    f1s = np.zeros(n_classes)
    for c in range(n_classes):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP
        f1s[c] = 2*TP / max(2*TP + FP + FN, 1e-10)
    return f1s

def nested_cv_classify(X, y, param_grid, outer_k=5, inner_k=3):
    outer_cv = StratifiedKFold(outer_k, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(inner_k, shuffle=True, random_state=0)
    outer_scores = []
    for tr, te in outer_cv.split(X, y):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(50)),
            ('logisticregression', LogisticRegression(max_iter=2000, multi_class='multinomial'))
        ])
        gs = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy')
        gs.fit(X[tr], y[tr])
        outer_scores.append(gs.score(X[te], y[te]))
    return np.array(outer_scores)

def top_gene_loadings(pca, n_top=20):
    top_pc1 = np.argsort(np.abs(pca.components_[0]))[-n_top:][::-1]
    top_pc2 = np.argsort(np.abs(pca.components_[1]))[-n_top:][::-1]
    return top_pc1, top_pc2

# MP2 References

def roc_curve_scratch(y_true, y_scores):
    thresholds = np.sort(np.unique(y_scores))[::-1]
    P = y_true.sum()
    N = (1-y_true).sum()
    fprs, tprs = [0], [0]
    for t in thresholds:
        yp = (y_scores >= t).astype(int)
        TP = ((yp==1)&(y_true==1)).sum()
        FP = ((yp==1)&(y_true==0)).sum()
        fprs.append(FP/N); tprs.append(TP/P)
    fprs.append(1); tprs.append(1)
    auroc = abs(np.trapz(tprs, fprs))
    return np.array(fprs), np.array(tprs), auroc

def pr_curve_scratch(y_true, y_scores):
    thresholds = np.sort(np.unique(y_scores))[::-1]
    P = y_true.sum()
    precisions, recalls = [1], [0]
    for t in thresholds:
        yp = (y_scores >= t).astype(int)
        TP = ((yp==1)&(y_true==1)).sum()
        FP = ((yp==1)&(y_true==0)).sum()
        precisions.append(TP/max(TP+FP,1e-10))
        recalls.append(TP/max(P,1e-10))
    auprc = abs(np.trapz(precisions, recalls))
    return np.array(recalls), np.array(precisions), auprc

def nested_cv_binary(X, y, outer_k=5, inner_k=3):
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    outer_cv = StratifiedKFold(outer_k, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(inner_k, shuffle=True, random_state=0)
    param_grid = {'n_estimators': [50, 100], 'max_depth': [3, 5, None]}
    auroc_scores, auprc_scores = [], []
    scaler = StandardScaler()
    for tr, te in outer_cv.split(X, y):
        X_tr_sc = scaler.fit_transform(X[tr])
        X_te_sc = scaler.transform(X[te])
        gs = GridSearchCV(
            RandomForestClassifier(class_weight='balanced', random_state=42),
            param_grid, cv=inner_cv, scoring='roc_auc')
        gs.fit(X_tr_sc, y[tr])
        prob = gs.predict_proba(X_te_sc)[:, 1]
        _, _, auroc = roc_curve_scratch(y[te], prob)
        _, _, auprc = pr_curve_scratch(y[te], prob)
        auroc_scores.append(auroc)
        auprc_scores.append(auprc)
    return np.array(auroc_scores), np.array(auprc_scores)

def rf_feature_importance(X, y, feat_names):
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X, y)
    order = np.argsort(rf.feature_importances_)[::-1]
    return [(feat_names[i], rf.feature_importances_[i]) for i in order[:3]]
-->